# ITM-390: ASD Screening in Children - Exploratory Data Analysis (EDA)
**Group 10** · American University of Phnom Penh · Advisor: Kuntha PIN  
*Comparative Analysis of Machine Learning Models for Autism Spectrum Disorder Screening in Children*

### Notebook Objectives:
1. Load raw dataset (`Autism-Child-Data.arff` / `.csv`).
2. Inspect data types, distributions, and missing values (`?`).
3. **Verify Critical Risk #1 (Target Leakage):** Prove that `result` is deterministically derived from `A1`..`A10` and leaks `Class/ASD`.
4. Explore demographic distributions (Age, Gender, Ethnicity, Jaundice, Family ASD history).
5. Conclude on preprocessing strategy for Phase 2.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetics
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 120

# Project imports
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.preprocessing import load_raw_data, clean_raw_data, LEAKAGE_COL, TARGET_COL
print('Project modules imported successfully!')

## 1. Load Raw Dataset

In [ ]:
raw_path = project_root / 'data' / 'raw' / 'Autism-Child-Data.arff'
df_raw = load_raw_data(raw_path)
print(f'Raw dataset shape: {df_raw.shape} (292 instances, 21 attributes)')
df_raw.head(5)

## 2. Target Leakage Verification (Critical Pre-Work #1)
Per Section 3 of the plan, the `result` column is the sum of answers `A1` to `A10`. If `result >= 7`, the target is assigned as ASD.

In [ ]:
# Calculate the sum of A1 to A10 answers
q_cols = [f'A{i}_Score' for i in range(1, 11)]
calc_sum = df_raw[q_cols].astype(int).sum(axis=1)
reported_result = df_raw['result'].astype(int)

is_identical = (calc_sum == reported_result).all()
print(f'Does `result` strictly equal the sum of A1..A10? {is_identical}')

# Check relationship between result score and target Class/ASD
df_raw['calc_result'] = calc_sum
cross_tab = pd.crosstab(df_raw['calc_result'], df_raw['Class/ASD'])
print('\nCross-tabulation of Score vs Class/ASD:')
print(cross_tab)

In [ ]:
# Plot score distribution colored by ASD class
plt.figure(figsize=(8, 4))
sns.histplot(data=df_raw, x='calc_result', hue='Class/ASD', multiple='stack', discrete=True, palette={'NO': 'steelblue', 'YES': 'crimson'})
plt.axvline(6.5, color='black', linestyle='--', label='Cutoff Threshold (>= 7)')
plt.title('Evidence of Deterministic Target Leakage via `result` Column')
plt.xlabel('Sum of Q-CHAT-10 Scores (A1 to A10)')
plt.ylabel('Number of Children')
plt.legend()
plt.tight_layout()
plt.show()

> **Decision:** The plot confirms that `result >= 7` strictly corresponds to `Class/ASD = YES` with 100% precision. Keeping `result` as a feature would cause trivial target leakage. **`result` must be dropped from all feature matrices.**

## 3. Data Cleaning and Missing Value Profiling

In [ ]:
df_clean = clean_raw_data(df_raw, drop_leakage=True)
print(f'Cleaned dataset shape: {df_clean.shape}')

# Missing values inspection
missing = df_clean.isna().sum()
missing = missing[missing > 0]
print('\nFeatures with missing values (formerly `?`):')
print(missing)

## 4. Class Balance & Demographics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Class distribution
sns.countplot(data=df_clean, x='asd_target', ax=axes[0], palette=['steelblue', 'crimson'])
axes[0].set_xticklabels(['No ASD (0)', 'ASD (1)'])
axes[0].set_title(f'Class Balance: {df_clean["asd_target"].value_counts().to_dict()}')
axes[0].set_ylabel('Count')

# Age distribution
sns.histplot(data=df_clean, x='age', hue='asd_target', multiple='stack', ax=axes[1], discrete=True, palette=['steelblue', 'crimson'])
axes[1].set_title('Age Distribution (Years 4 - 11)')
axes[1].legend(['ASD (1)', 'No ASD (0)'])

plt.tight_layout()
plt.show()

## 5. EDA Summary & Next Steps
- `result` column successfully verified as leakage and eliminated.
- Constant column `age_desc` (4-11 years) removed.
- Missing values (`ethnicity`, `relation`, `age`) identified for median/most-frequent imputation inside the `ColumnTransformer` pipeline.
- Proceed to `02_models.ipynb` for cross-validation and hyperparameter tuning.